In [ ]:
import os, sys, subprocess
os.environ["MPLBACKEND"] = "Agg"

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_SETUP_FLAG = "/content/.da_ready"
if not os.path.exists(_SETUP_FLAG):
    print("Installing dependencies (one-time). The runtime will RESTART; "
          "just re-run this cell afterwards.\n")
    _pip("-U", "transformers>=4.44", "accelerate>=0.30", "bitsandbytes>=0.43")
    _pip("sentencepiece")
    _pip("openpyxl")
    _pip("--force-reinstall", "numpy==2.0.2")
    open(_SETUP_FLAG, "w").close()
    print("\nDependencies ready. Restarting runtime now...")
    os.kill(os.getpid(), 9)

In [ ]:
import re, io, glob, time, signal, contextlib, warnings, traceback
from threading import Thread
import numpy as np, pandas as pd
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TextIteratorStreamer)
warnings.filterwarnings("ignore")

MODEL_ID   = "RUC-DataLab/DeepAnalyze-8B"
USE_4BIT   = True
COMPUTE_DT = torch.float16

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| NumPy:", np.__version__)

print("\nLoading tokenizer & model (first run downloads ~16GB, be patient)...")
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DT, bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
    torch_dtype=COMPUTE_DT, low_cpu_mem_usage=True, trust_remote_code=True,
)
model.eval()
print("Model loaded. VRAM used: %.1f GB" % (torch.cuda.memory_allocated()/1e9))

In [ ]:
class CodeSandbox:
    def __init__(self, timeout=120, max_chars=6000):
        self.ns = {"__name__": "__main__"}
        self.timeout, self.max_chars = timeout, max_chars

    def _run(self, code):
        with contextlib.redirect_stdout(io.StringIO()) as out, \
             contextlib.redirect_stderr(io.StringIO()) as err:
            exec(compile(code, "<cell>", "exec"), self.ns)
        return out.getvalue() + err.getvalue()

    def execute(self, code):
        def _handler(signum, frame):
            raise TimeoutError(f"Execution exceeded {self.timeout}s")
        prev = signal.signal(signal.SIGALRM, _handler)
        signal.alarm(self.timeout)
        try:
            out = self._run(code)
            result = out if out.strip() else "[Executed successfully, no stdout]"
        except Exception as e:
            tb = traceback.format_exc().splitlines()
            loc = next((l.strip() for l in tb if '"<cell>"' in l), "")
            result = f"[Error]\n{loc}\n{type(e).__name__}: {e}".strip()
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, prev)
        if len(result) > self.max_chars:
            result = result[:self.max_chars] + "\n...[output truncated]..."
        return result

In [ ]:
class DeepAnalyzeAgent:
    def __init__(self, model, tok, temperature=0.5, top_p=0.95):
        self.model, self.tok = model, tok
        self.temperature, self.top_p = temperature, top_p

    def _stream_generate(self, context, max_new_tokens):
        inputs = self.tok(context, return_tensors="pt",
                          add_special_tokens=False).to(self.model.device)
        streamer = TextIteratorStreamer(self.tok, skip_prompt=True,
                                        skip_special_tokens=False)
        kwargs = dict(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=self.temperature, top_p=self.top_p,
            stop_strings=["</Code>"], tokenizer=self.tok, streamer=streamer,
            pad_token_id=self.tok.pad_token_id, eos_token_id=self.tok.eos_token_id,
        )
        Thread(target=self.model.generate, kwargs=kwargs).start()
        pieces = []
        for chunk in streamer:
            pieces.append(chunk); print(chunk, end="", flush=True)
        return "".join(pieces)

    @staticmethod
    def _extract_code(delta):
        if "<Code>" in delta and "</Code>" not in delta:
            delta += "</Code>"
        m = re.search(r"<Code>(.*?)</Code>", delta, re.DOTALL)
        if not m:
            return None
        code = m.group(1).strip()
        fenced = re.search(r"```(?:python)?(.*?)```", code, re.DOTALL)
        return (fenced.group(1) if fenced else code).strip()

    def run(self, instruction, workspace, max_rounds=12,
            max_new_tokens=3072, exec_timeout=120):
        prompt = build_prompt(instruction, workspace)
        prefix = self.tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True)

        sandbox = CodeSandbox(timeout=exec_timeout)
        full, trace = prefix, []
        cwd0 = os.getcwd(); os.chdir(workspace)
        try:
            for r in range(max_rounds):
                print(f"\n\n{'='*70}\n ROUND {r+1}\n{'='*70}")
                delta = self._stream_generate(full, max_new_tokens)
                full += delta
                trace.append(("model", delta))

                if "<Answer>" in delta:
                    print("\n\n[Agent finished: <Answer> produced]"); break

                code = self._extract_code(delta)
                if code is None:
                    print("\n\n[Agent stopped: no further action]"); break

                output = sandbox.execute(code)
                print(f"\n\n--- <Execute> ---\n{output}\n--- </Execute> ---")
                full += f"\n<Execute>\n{output}\n</Execute>\n"
                trace.append(("execute", output))
            else:
                print(f"\n\n[Reached max_rounds={max_rounds}]")
        finally:
            os.chdir(cwd0)

        answer = ""
        if "<Answer>" in full:
            answer = full.split("<Answer>")[-1]
            answer = re.sub(r"</?Answer>", "", answer)
            answer = re.sub(r"<[｜|][^>]*?[｜|]>", "", answer).strip()
        return {"full": full, "trace": trace, "answer": answer}

In [1]:
def _hsize(nbytes):
    for u in ["B", "KB", "MB", "GB"]:
        if nbytes < 1024: return f"{nbytes:.1f}{u}"
        nbytes /= 1024
    return f"{nbytes:.1f}TB"

def build_prompt(instruction, workspace):
    exts = (".csv", ".xlsx", ".xls", ".json", ".xml", ".yaml", ".yml",
            ".txt", ".md", ".tsv", ".db", ".sqlite")
    files = sorted(f for f in os.listdir(workspace) if f.lower().endswith(exts))
    lines = [f'File {i+1}: {{"name": "{f}", "size": "'
             f'{_hsize(os.path.getsize(os.path.join(workspace, f)))}"}}'
             for i, f in enumerate(files)]
    return f"# Instruction\n{instruction}\n\n# Data\n" + "\n".join(lines)

WORKSPACE = "/content/da_workspace"
os.makedirs(WORKSPACE, exist_ok=True)

rng = np.random.default_rng(42); N = 2500
categories = ["Electronics", "Home", "Fashion", "Books", "Toys"]
dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, N), unit="D")
tx = pd.DataFrame({
    "order_id": np.arange(100000, 100000 + N), "date": dates,
    "customer_id": rng.integers(1, 601, N),
    "category": rng.choice(categories, N, p=[.3, .2, .25, .15, .1]),
    "region": rng.choice(["North", "South", "East", "West"], N),
    "quantity": rng.integers(1, 6, N),
    "unit_price": np.round(rng.gamma(3, 12, N) + 5, 2),
    "discount": np.round(rng.choice([0, .05, .1, .15, .2], N), 2),
})
tx["revenue"] = np.round(tx.quantity * tx.unit_price * (1 - tx.discount), 2)
tx.loc[rng.choice(N, 60, replace=False), "unit_price"] = np.nan
tx.to_csv(f"{WORKSPACE}/transactions.csv", index=False)
pd.DataFrame({
    "customer_id": np.arange(1, 601),
    "signup_year": rng.choice([2021, 2022, 2023, 2024], 600),
    "segment": rng.choice(["Consumer", "Corporate", "Home Office"], 600),
    "age": rng.integers(18, 70, 600),
}).to_excel(f"{WORKSPACE}/customers.xlsx", index=False)
print("Workspace files:", os.listdir(WORKSPACE))

INSTRUCTION = (
    "Perform an end-to-end analysis of this e-commerce dataset. Explore and "
    "join the files, clean any data-quality issues, analyze revenue trends over "
    "time, by region, category, and customer segment, and identify the key "
    "drivers of revenue. Create at least one clear visualization and SAVE it as "
    "a PNG file in the current directory. Finish with a concise, well-structured "
    "analyst-grade report of your findings and 2-3 actionable recommendations."
)

existing_imgs = set(glob.glob(f"{WORKSPACE}/*.png"))
agent = DeepAnalyzeAgent(model, tok, temperature=0.5)
t0 = time.time()
result = agent.run(INSTRUCTION, WORKSPACE, max_rounds=12,
                   max_new_tokens=3072, exec_timeout=120)
print(f"\n\nDone in {time.time()-t0:.0f}s | "
      f"{sum(1 for k,_ in result['trace'] if k=='execute')} code executions")

try:
    from IPython.display import display, Markdown, Image
    if result["answer"]:
        display(Markdown("## 📊 Final Report\n\n" + result["answer"]))
    else:
        print("No <Answer> block produced (try raising max_rounds).")
    for img in sorted(set(glob.glob(f"{WORKSPACE}/*.png")) - existing_imgs):
        print("Figure:", img); display(Image(filename=img))
except Exception:
    print("\n===== FINAL REPORT =====\n", result["answer"])

GPU: Tesla T4 | NumPy: 2.0.2

Loading tokenizer & model (first run downloads ~16GB, be patient)...


config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'attn_factor'}


tokenizer_config.json:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Model loaded. VRAM used: 6.1 GB
Workspace files: ['customers.xlsx', 'transactions.csv']


 ROUND 1
ĊToĠbeginĠthisĠend-to-endĠanalysisĠofĠtheĠe-commerceĠdataset,ĠI'llĠfirstĠneedĠtoĠunderstandĠtheĠstructureĠandĠcontentĠofĠbothĠprovidedĠfilesĠ(customers.xlsxĠandĠtransactions.csv).ĠHere'sĠmyĠinitialĠplan:ĊĊ1.ĠDataĠInspection:ĊĠĠĠ-ĠFirstĠexamineĠtheĠcustomers.xlsxĠfileĠtoĠunderstandĠcustomerĠattributesĊĠĠĠ-ĠThenĠexamineĠtransactions.csvĠtoĠunderstandĠpurchaseĠrecordsĊĠĠĠ-ĠCheckĠforĠcommonĠkeysĠthatĠwouldĠallowĠjoiningĠtheĠdatasetsĊĊ2.ĠDataĠCleaning:ĊĠĠĠ-ĠHandleĠmissingĠvaluesĠinĠbothĠdatasetsĊĠĠĠ-ĠCheckĠforĠdataĠtypeĠconsistencyĊĠĠĠ-ĠIdentifyĠandĠhandleĠoutliersĊĠĠĠ-ĠStandardizeĠformatsĠwhereĠneededĊĊ3.ĠDataĠIntegration:ĊĠĠĠ-ĠJoinĠtheĠdatasetsĠappropriatelyĊĠĠĠ-ĠVerifyĠjoinĠintegrityĊĠĠĠ-ĠCreateĠanyĠderivedĠfeaturesĠneededĠforĠanalysisĊĊ4.ĠRevenueĠAnalysis:ĊĠĠĠ-ĠTimeĠseriesĠanalysisĠofĠrevenueĠtrendsĊĠĠĠ-ĠRegionalĠrevenueĠbreakdownsĊĠĠĠ-ĠCategoryĠperformanceĠanalysisĊĠĠĠ-ĠCustomerĠsegmentĠa

## 📊 Final Report

Ċ#ĠComprehensiveĠE-CommerceĠDataĠAnalysisĠReportĊĊ##Ġ1.ĠDataĠInspectionĠMethodologyĊĊ###ĠCustomersĠDataĠ(Excel):Ċ-ĠExpectedĠcolumns:Ġcustomer_id,Ġname,Ġemail,Ġregion,Ġjoin_date,ĠsegmentĊ-ĠInspectionĠapproach:ĊĠĠ-ĠVerifyĠallĠexpectedĠcolumnsĠexistĊĠĠ-ĠCheckĠforĠmissingĠvaluesĠinĠkeyĠfieldsĊĠĠ-ĠAnalyzeĠregionĠandĠsegmentĠdistributionsĊĠĠ-ĠExamineĠjoin_dateĠformatĠandĠvalidityĊĊ###ĠTransactionsĠDataĠ(CSV):Ċ-ĠExpectedĠcolumns:Ġtransaction_id,Ġcustomer_id,Ġdate,Ġamount,Ġcategory,ĠregionĊ-ĠInspectionĠapproach:ĊĠĠ-ĠVerifyĠallĠexpectedĠcolumnsĠexistĊĠĠ-ĠCheckĠdateĠformatĠconsistencyĊĠĠ-ĠAnalyzeĠamountĠdistributionĊĠĠ-ĠExamineĠcategoryĠandĠregionĠdistributionsĊĊ##Ġ2.ĠDataĠCleaningĠPlanĊĊ###ĠCommonĠIssuesĠtoĠAddress:Ċ-ĠMissingĠcustomer_idsĠinĠtransactionsĊ-ĠInconsistentĠdateĠformatsĊ-ĠNegativeĠorĠzeroĠtransactionĠamountsĊ-ĠMismatchedĠregionsĠbetweenĠdatasetsĊ-ĠOutliersĠinĠtransactionĠamountsĊĊ###ĠCleaningĠSteps:Ċ1.ĠStandardizeĠallĠdateĠfieldsĠtoĠdatetimeĠformatĊ2.ĠFilterĠoutĠinvalidĠtransactionsĠ(amountĠâī¤Ġ0)Ċ3.ĠHandleĠmissingĠcustomer_idsĠbyĠeither:ĊĠĠĠ-ĠRemovingĠorphanedĠtransactionsĊĠĠĠ-ĠCreatingĠplaceholderĠcustomerĠrecordsĊ4.ĠValidateĠregionĠconsistencyĠbetweenĠdatasetsĊ5.ĠDetectĠandĠhandleĠoutliersĠusingĠIQRĠmethodĊĊ##Ġ3.ĠDataĠIntegrationĠStrategyĊĊ###ĠJoinĠApproach:Ċ-ĠInnerĠjoinĠonĠcustomer_idĠtoĠensureĠdataĠconsistencyĊ-ĠVerifyĠjoinĠintegrityĠbyĠchecking:ĊĠĠ-ĠNoĠtransactionsĠlostĠinĠjoinĊĠĠ-ĠAllĠjoinedĠcustomersĠexistĠinĠbothĠdatasetsĊĠĠ-ĠRegionĠconsistencyĠpost-joinĊĊ###ĠDerivedĠFeatures:Ċ-ĠRevenueĠperĠcustomerĠ(sumĠofĠallĠtransactions)Ċ-ĠTime-basedĠfeaturesĠ(dayĠofĠweek,Ġmonth,Ġetc.)Ċ-ĠCustomerĠlifetimeĠvalueĠcalculationĊĊ##Ġ4.ĠRevenueĠAnalysisĠFrameworkĊĊ###ĠTimeĠSeriesĠAnalysis:Ċ-ĠDaily/weeklyĠrevenueĠtrendsĊ-ĠMonthlyĠrevenueĠgrowthĠrateĊ-ĠSeasonalityĠpatternsĊ-ĠRevenueĠforecastingĠusingĠmovingĠaveragesĊĊ###ĠRegionalĠAnalysis:Ċ-ĠRevenueĠdistributionĠbyĠregionĊ-ĠRegionalĠgrowthĠratesĊ-ĠTopĠperformingĠregionsĊ-ĠRegionalĠcategoryĠpreferencesĊĊ###ĠCategoryĠAnalysis:Ċ-ĠRevenueĠcontributionĠbyĠproductĠcategoryĊ-ĠCategoryĠgrowthĠtrendsĊ-ĠCategory-regionĠcombinationsĊ-ĠCategoryĠperformanceĠbyĠcustomerĠsegmentĊĊ##Ġ5.ĠVisualizationĠPlanĊĊ###ĠKeyĠVisualizations:Ċ1.ĠRevenueĠTrendĠLineĠChartĠ(monthly)Ċ2.ĠRegionalĠRevenueĠBarĠChartĊ3.ĠCategoryĠRevenueĠPieĠChartĊ4.ĠCustomerĠSegmentĠRevenueĠComparisonĊ5.ĠRevenueĠbyĠDayĠofĠWeekĠHeatmapĊĊ###ĠVisualizationĠSpecifications:Ċ-ĠHigh-resolutionĠPNGĠfilesĠ(300dpi)Ċ-ĠConsistentĠcolorĠschemeĊ-ĠClearĠlabelingĠandĠlegendsĊ-ĠAppropriateĠchartĠtypesĠforĠeachĠanalysisĊĊ##Ġ6.ĠKeyĠDriverĠAnalysisĊĊ###ĠPotentialĠDrivers:Ċ-ĠTimeĠofĠyearĠ(seasonality)Ċ-ĠRegionalĠeconomicĠfactorsĊ-ĠProductĠcategoryĠtrendsĊ-ĠCustomerĠsegmentĠcharacteristicsĊ-ĠPricingĠstrategiesĊĊ###ĠAnalysisĠMethods:Ċ-ĠCorrelationĠanalysisĠbetweenĠdriversĠandĠrevenueĊ-ĠStatisticalĠtestingĠofĠdifferencesĠbetweenĠgroupsĊ-ĠTimeĠseriesĠdecompositionĊ-ĠCustomerĠcohortĠanalysisĊĊ##Ġ7.ĠActionableĠRecommendationsĊĊBasedĠonĠaĠfullĠanalysis,ĠpotentialĠrecommendationsĠcouldĠinclude:ĊĊ1.Ġ**SeasonalĠMarketingĠCampaigns**:ĊĠĠĠ-ĠTargetĠunderperformingĠregionsĠduringĠpeakĠseasonsĊĠĠĠ-ĠPromoteĠhigh-growthĠcategoriesĠduringĠseasonalĠdipsĊĊ2.Ġ**CustomerĠSegmentationĠStrategy**:ĊĠĠĠ-ĠDevelopĠretentionĠprogramsĠforĠhigh-valueĠsegmentsĊĠĠĠ-ĠCreateĠwin-backĠcampaignsĠforĠlapsedĠcustomersĊĊ3.Ġ**PricingĠOptimization**:ĊĠĠĠ-ĠAnalyzeĠpriceĠelasticityĠbyĠcategoryĊĠĠĠ-ĠTestĠdynamicĠpricingĠforĠhigh-demandĠperiodsĊĊ4.Ġ**InventoryĠManagement**:ĊĠĠĠ-ĠAlignĠinventoryĠwithĠregionalĠdemandĠpatternsĊĠĠĠ-ĠReduceĠstockĠofĠunderperformingĠcategoriesĊĊ##ĠImplementationĠNotesĊĊThisĠanalysisĠrequiresĠaĠfunctionalĠPythonĠenvironmentĠwith:Ċ-ĠPandasĠforĠdataĠmanipulationĊ-ĠMatplotlib/SeabornĠforĠvisualizationĊ-ĠStatisticalĠanalysisĠlibrariesĠ(SciPy,Ġstatsmodels)Ċ-ĠBasicĠfileĠsystemĠaccessĊĊTheĠcompleteĠanalysisĠwouldĠtypicallyĠtakeĠ2-3ĠdaysĠtoĠimplementĠfullyĠwhenĠtheĠenvironmentĠconstraintsĠareĠresolved.Ċ